# B3 — ResNet50 + Vanilla EDL + KL — integration release v3.3


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install "torch>=2.2" "torchvision>=0.17" scikit-learn pandas numpy matplotlib pillow tqdm thop scipy

In [ ]:
import os, gc, json, math, time, random, warnings, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, StratifiedShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import label_binarize
from scipy.stats import wilcoxon
from tqdm.auto import tqdm
from thop import profile

warnings.filterwarnings("ignore")


In [ ]:
# Canonical project layout
MYDRIVE = Path('/content/drive/MyDrive')
THESIS_ROOT_CANDIDATES = [
    MYDRIVE / 'Thesis_Work/LIDC_Thesis',
    MYDRIVE / 'Thesis_Work/LIDC_Thesis',     # preprocessing-v3 root in the supplied notebook
]

def choose_thesis_root():
    for root in THESIS_ROOT_CANDIDATES:
        if (root/'LIDC/processed/metadata/lidc_patches_metadata.csv').exists():
            return root
    return THESIS_ROOT_CANDIDATES[0]

THESIS_ROOT = choose_thesis_root()
LIDC_ROOT = THESIS_ROOT / 'LIDC'
PATCH_ROOT = LIDC_ROOT / 'processed' / 'patches'
METADATA_ROOT = LIDC_ROOT / 'processed' / 'metadata'
PREPROCESS_CONFIG = LIDC_ROOT / 'preprocessing' / 'configs' / 'preprocessing_config.yaml'
CLASS_MAPPING_JSON = LIDC_ROOT / 'preprocessing' / 'configs' / 'class_mapping.json'
METADATA_CSV = METADATA_ROOT / 'lidc_patches_metadata.csv'
FOLD_ASSIGNMENTS_CSV = METADATA_ROOT / 'fold_assignments.csv'
FOLD_HASH_TXT = METADATA_ROOT / 'fold_assignments.sha256'
SHARED_PIPELINE_CONFIG = METADATA_ROOT / 'shared_pipeline_config.json'

# B3 experiment folder: all generated artifacts stay below this directory.
B3_ROOT = THESIS_ROOT / 'experiments' / 'B3_ResNet50_Vanilla_EDL_KL'
CONFIG_ROOT = B3_ROOT/'configs'; SPLIT_ROOT=B3_ROOT/'splits'; CHECKPOINT_ROOT=B3_ROOT/'checkpoints'
PREDICTION_ROOT=B3_ROOT/'predictions'; METRICS_ROOT=B3_ROOT/'metrics'; PLOT_ROOT=B3_ROOT/'plots'
LOG_ROOT=B3_ROOT/'logs'; REPORT_ROOT=B3_ROOT/'reports'; XAI_ROOT=B3_ROOT/'xai'; ABLATION_ROOT=B3_ROOT/'ablations'
CLASSIFICATION_METRICS=METRICS_ROOT/'classification'; CALIBRATION_METRICS=METRICS_ROOT/'calibration'
UNCERTAINTY_METRICS=METRICS_ROOT/'uncertainty'; EFFICIENCY_METRICS=METRICS_ROOT/'efficiency'
TRAINING_PLOTS=PLOT_ROOT/'training_curves'; CM_PLOTS=PLOT_ROOT/'confusion_matrices'; ROC_PLOTS=PLOT_ROOT/'roc_curves'
CALIBRATION_PLOTS=PLOT_ROOT/'calibration_diagrams'; RELIABILITY_PLOTS=PLOT_ROOT/'reliability_diagrams'; REJECTION_PLOTS=PLOT_ROOT/'rejection_curves'
GRADCAM_ROOT=XAI_ROOT/'gradcam'; ESM_ROOT=XAI_ROOT/'esm'; XAI_METRICS=XAI_ROOT/'metrics'; XAI_PLOTS=XAI_ROOT/'plots'

for p in [CONFIG_ROOT,SPLIT_ROOT,CHECKPOINT_ROOT,PREDICTION_ROOT,CLASSIFICATION_METRICS,CALIBRATION_METRICS,
          UNCERTAINTY_METRICS,EFFICIENCY_METRICS,TRAINING_PLOTS,CM_PLOTS,ROC_PLOTS,CALIBRATION_PLOTS,
          RELIABILITY_PLOTS,REJECTION_PLOTS,LOG_ROOT,REPORT_ROOT,GRADCAM_ROOT,ESM_ROOT,XAI_METRICS,XAI_PLOTS,ABLATION_ROOT]:
    p.mkdir(parents=True,exist_ok=True)
for fold in range(1,6):
    (SPLIT_ROOT/f'fold_{fold}').mkdir(parents=True,exist_ok=True)
    (CHECKPOINT_ROOT/f'fold_{fold}').mkdir(parents=True,exist_ok=True)

print('THESIS_ROOT:',THESIS_ROOT)
print('PATCH_ROOT:',PATCH_ROOT)
print('METADATA_CSV:',METADATA_CSV)
print('B3_ROOT:',B3_ROOT)


THESIS_ROOT: /content/drive/MyDrive/Thesis_Work/LIDC_Thesis
PATCH_ROOT: /content/drive/MyDrive/Thesis_Work/LIDC_Thesis/LIDC/processed/patches
METADATA_CSV: /content/drive/MyDrive/Thesis_Work/LIDC_Thesis/LIDC/processed/metadata/lidc_patches_metadata.csv
B3_ROOT: /content/drive/MyDrive/Thesis_Work/LIDC_Thesis/experiments/B3_ResNet50_Vanilla_EDL_KL


In [ ]:
CONFIG = {
    'config_id':'B3','backbone':'ResNet50','uq_method':'Vanilla EDL + KL',
    'num_classes':3,'class_names':['benign','indeterminate','malignant'],
    'seed':42,'n_splits':5,'validation_fraction_of_training_patients':0.10,
    'image_size':224,'batch_size':32,'num_workers':0,'max_epochs':50,'early_stopping_patience':10,
    'learning_rate':1e-4,'eta_min':1e-6,'weight_decay':0.01,'adam_beta1':0.9,'adam_beta2':0.999,
    'prior_weight_C':1.0,
    # The proposal specifies L_ECE + lambda_KL*L_KL but does not give a numeric lambda_KL.
    # 1.0 is retained as the explicit B3 run value and recorded in every manifest/checkpoint.
    'lambda_kl':1.0,
    'kl_anneal_epochs':10,
    'epsilon':1e-8,'ece_bins':15,'random_rejection_repeats':50,
    'augmentation':{'horizontal_flip':True,'vertical_flip':True,'rotation_degrees':15,'translation_fraction':0.10,
                    'elastic_deformation':False,'intensity_jitter':False},
    'preprocessing_expected':{'minimum_diameter_mm':3.0,'minimum_radiologists':3,'variance_normalizer':4.0,
                              'hu_low':-1000.0,'hu_high':400.0,'patch_range':[0.0,1.0]},
    # IMPORTANT: pilot_max_patients is loaded from shared_pipeline_config.json below.
    # No model notebook is allowed to define an independent pilot subset.
    'pilot_max_patients':None,
    'ablation_prior_C_values':[0.5,1.0,2.0], 'ablation_loss_modes':['ece_only','ece_plus_kl'],
    'xai_max_examples_per_fold':30,'xai_metric_steps':20,
}
with open(CONFIG_ROOT/'B3_config.json','w') as f: json.dump(CONFIG,f,indent=2)

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED']=str(seed); random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    try: torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception: pass
seed_everything(CONFIG['seed'])
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE)


Device: cuda


In [ ]:

LABEL_MAP={'benign':0,'indeterminate':1,'malignant':2}
INV_LABEL_MAP={v:k for k,v in LABEL_MAP.items()}

# ---------------- Shared preprocessing contract ----------------
for required_file in [METADATA_CSV, FOLD_ASSIGNMENTS_CSV, FOLD_HASH_TXT, SHARED_PIPELINE_CONFIG]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Missing shared preprocessing artifact: {required_file}\n"
            "Run the corrected master preprocessing notebook first. "
            "B3 must not recreate labels, folds, or pilot subsets locally."
        )

with open(SHARED_PIPELINE_CONFIG) as f:
    shared_cfg=json.load(f)
CONFIG['pilot_max_patients']=shared_cfg.get('PILOT_MAX_PATIENTS', None)
CONFIG['preprocessing_version']=shared_cfg.get('preprocessing_version','unknown')
CONFIG['crop_geometry']=shared_cfg.get('crop_geometry',{})

def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

actual_fold_hash=sha256_file(FOLD_ASSIGNMENTS_CSV)
expected_fold_hash=FOLD_HASH_TXT.read_text().strip().split()[0]
if actual_fold_hash != expected_fold_hash:
    raise RuntimeError(
        f"Fold assignment hash mismatch.\nExpected: {expected_fold_hash}\nActual:   {actual_fold_hash}"
    )
CONFIG['fold_assignments_sha256']=actual_fold_hash

metadata=pd.read_csv(METADATA_CSV)
fold_assignments=pd.read_csv(FOLD_ASSIGNMENTS_CSV)

required=[
    'nodule_id','patient_id','label_name','label_id','patch_path',
    'n_radiologists','diameter_mm','variance_normalized','alignment_mask'
]
missing=[c for c in required if c not in metadata.columns]
if missing:
    raise ValueError(f'Authoritative preprocessing metadata is missing required columns: {missing}')

if set(['patient_id','fold'])-set(fold_assignments.columns):
    raise ValueError("fold_assignments.csv must contain patient_id and fold columns.")

metadata=metadata.rename(columns={
    'label_id':'label','label_name':'class_name','patch_path':'image_path'
}).copy()
metadata['label']=pd.to_numeric(metadata['label'],errors='raise').astype(int)
metadata['class_name']=metadata['class_name'].astype(str).str.lower()
metadata['patient_id']=metadata['patient_id'].astype(str)
fold_assignments['patient_id']=fold_assignments['patient_id'].astype(str)
fold_assignments['fold']=pd.to_numeric(fold_assignments['fold'],errors='raise').astype(int)

# Never infer labels from folder names. Folder names are storage only.
def resolve_npy_path(r):
    p=Path(str(r.image_path))
    if p.exists() and p.suffix.lower()=='.npy':
        return str(p)
    # Recovery uses nodule_id only; class_name is deliberately NOT used.
    candidate=PATCH_ROOT/f'{r.nodule_id}.npy'
    if candidate.exists():
        return str(candidate)
    # Backward-compatible search by filename, without using folder as a label source.
    hits=list(PATCH_ROOT.rglob(f'{r.nodule_id}.npy'))
    return str(hits[0]) if len(hits)==1 else str(p)

metadata['image_path']=metadata.apply(resolve_npy_path,axis=1)

# Fold is authoritative from the single shared fold file.
metadata=metadata.drop(columns=['fold'],errors='ignore').merge(
    fold_assignments[['patient_id','fold']].drop_duplicates(),
    on='patient_id',how='left',validate='many_to_one'
)
if metadata['fold'].isna().any():
    bad=metadata.loc[metadata['fold'].isna(),'patient_id'].unique()[:10]
    raise RuntimeError(f"Patients missing from fold_assignments.csv: {bad}")
metadata['fold']=metadata['fold'].astype(int)

assert set(metadata['label'].unique()).issubset({0,1,2})
assert set(metadata['fold'].unique())==set(range(1,CONFIG['n_splits']+1)), "Expected outer folds 1..5."
assert (metadata['n_radiologists']>=3).all()
assert (metadata['diameter_mm']>=3.0).all()
assert metadata['variance_normalized'].between(0,1).all()
assert metadata.groupby('patient_id')['fold'].nunique().max()==1, "Patient leakage across outer folds."

missing_files=[p for p in metadata['image_path'] if not Path(p).is_file()]
if missing_files:
    raise FileNotFoundError(f'{len(missing_files)} patch files are missing. First: {missing_files[0]}')

# Single shared pilot switch: preprocessing is responsible for selecting/marking the effective cohort.
# B3 only verifies the shared setting; it does not choose its own patients.
if CONFIG['pilot_max_patients'] is not None:
    if 'pilot_included' not in metadata.columns:
        raise RuntimeError("Pilot mode requires preprocessing metadata column 'pilot_included'.")
    metadata=metadata[metadata['pilot_included'].astype(bool)].reset_index(drop=True)

if metadata.empty:
    raise RuntimeError("Authoritative metadata contains zero usable rows.")

# Persist the FINAL resolved configuration, including shared preprocessing provenance.
with open(CONFIG_ROOT/'B3_config.json','w') as f:
    json.dump(CONFIG,f,indent=2)

metadata.to_csv(REPORT_ROOT/'B3_effective_metadata.csv',index=False)
print("Fold SHA256:",actual_fold_hash)
print("Pilot max patients:",CONFIG['pilot_max_patients'])
print('Patches:',len(metadata),'Patients:',metadata.patient_id.nunique())
display(metadata.groupby('class_name').size().rename('patches').to_frame())


Fold SHA256: 6579bbae4001911dbeb74213340e7846c3dcc2d03f915ed501c0ec339446a060
Pilot max patients: None
Patches: 48 Patients: 21


,patches
class_name,
benign,5
indeterminate,33
malignant,10


In [ ]:
bad=[]
for p in tqdm(metadata['image_path'],desc='Checking .npy patches'):
    try:
        a=np.load(p,mmap_mode='r')
        if a.shape!=(224,224) or not np.isfinite(a).all(): raise ValueError(f'shape={a.shape}')
    except Exception as e: bad.append({'patch_path':p,'error':repr(e)})
if bad:
    pd.DataFrame(bad).to_csv(LOG_ROOT/'unreadable_patches.csv',index=False)
    raise RuntimeError(f'{len(bad)} unreadable/invalid patches; see logs/unreadable_patches.csv')
print('All preprocessing .npy patches passed integrity checks.')


Checking .npy patches:   0%|          | 0/48 [00:00<?, ?it/s]

All preprocessing .npy patches passed integrity checks.


In [ ]:
fold_frames={}
manifest=[]

def make_train_val(trainval, fold_number):
    """Deterministic patient-disjoint stratified validation split."""
    from collections import Counter
    from sklearn.model_selection import StratifiedShuffleSplit
    required_cols={"patient_id","label"}
    missing_cols=required_cols-set(trainval.columns)
    if missing_cols:
        raise RuntimeError(
            f"Fold {fold_number}: train/validation metadata contract mismatch; "
            f"missing={sorted(missing_cols)}, available={list(trainval.columns)}"
        )
    patient_rows=[]
    for pid,g in trainval.groupby("patient_id",sort=True):
        counts=g["label"].value_counts()
        mx=counts.max()
        dominant=int(sorted(counts[counts==mx].index.astype(int))[0])
        patient_rows.append((str(pid),dominant))
    ptab=pd.DataFrame(patient_rows,columns=["patient_id","patient_label"])
    labels=ptab["patient_label"].to_numpy(int)
    counts=Counter(labels); n=len(ptab); k=len(counts)
    if k<2 or min(counts.values())<2:
        raise RuntimeError(
            f"Fold {fold_number}: patient-level stratification impossible; "
            f"patient-class counts={dict(counts)}."
        )
    target=max(1,int(round(CONFIG["validation_fraction_of_training_patients"]*n)))
    nval=max(target,k)
    nval=min(nval,n-k)
    if nval<k or n-nval<k:
        raise RuntimeError(
            f"Fold {fold_number}: no feasible patient-stratified validation split; "
            f"patient-class counts={dict(counts)}, patients={n}."
        )
    splitter=StratifiedShuffleSplit(
        n_splits=1,test_size=nval,random_state=int(CONFIG["seed"])+int(fold_number)
    )
    tri,vai=next(splitter.split(ptab["patient_id"],labels))
    trp=set(ptab.iloc[tri].patient_id); vap=set(ptab.iloc[vai].patient_id)
    tr=trainval[trainval.patient_id.astype(str).isin(trp)].copy()
    va=trainval[trainval.patient_id.astype(str).isin(vap)].copy()
    assert set(tr.patient_id.astype(str)).isdisjoint(set(va.patient_id.astype(str)))
    method=(f"patient_stratified_shuffle(target_n={target},actual_n={nval},"
            f"actual_fraction={nval/n:.3f})")
    return tr,va,method

for fold in range(1,CONFIG['n_splits']+1):
    te=metadata[metadata.fold==fold].copy()
    tv=metadata[metadata.fold!=fold].copy()
    tr,va,val_method=make_train_val(tv,fold)
    assert set(tr.patient_id).isdisjoint(va.patient_id)
    assert set(tr.patient_id).isdisjoint(te.patient_id)
    assert set(va.patient_id).isdisjoint(te.patient_id)

    fold_frames[fold]={
        'train':tr.reset_index(drop=True),
        'val':va.reset_index(drop=True),
        'test':te.reset_index(drop=True)
    }
    fd=SPLIT_ROOT/f'fold_{fold}'
    for name,frame in fold_frames[fold].items():
        frame.to_csv(fd/f'{name}.csv',index=False)
        manifest.append({
            'fold':fold,'split':name,
            'outer_fold_source':'fold_assignments.csv',
            'fold_assignments_sha256':actual_fold_hash,
            'validation_method':val_method if name!='test' else 'shared preprocessing fold',
            'patches':len(frame),'patients':frame.patient_id.nunique(),
            'benign':int((frame.label==0).sum()),
            'indeterminate':int((frame.label==1).sum()),
            'malignant':int((frame.label==2).sum())
        })

split_manifest=pd.DataFrame(manifest)
split_manifest.to_csv(SPLIT_ROOT/'all_folds_summary.csv',index=False)
display(split_manifest)


,fold,split,outer_fold_source,fold_assignments_sha256,validation_method,patches,patients,benign,indeterminate,malignant
0,1,train,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,"patient_stratified_shuffle(target_n=2,actual_n...",40,16,4,29,7
1,1,val,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,"patient_stratified_shuffle(target_n=2,actual_n...",6,3,0,4,2
2,1,test,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,shared preprocessing fold,2,2,1,0,1
3,2,train,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,"patient_stratified_shuffle(target_n=2,actual_n...",26,13,4,17,5
4,2,val,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,"patient_stratified_shuffle(target_n=2,actual_n...",3,3,0,2,1
5,2,test,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,shared preprocessing fold,19,5,1,14,4
6,3,train,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,"patient_stratified_shuffle(target_n=2,actual_n...",33,14,5,23,5
7,3,val,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,"patient_stratified_shuffle(target_n=2,actual_n...",9,3,0,5,4
8,3,test,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,shared preprocessing fold,6,4,0,5,1
9,4,train,fold_assignments.csv,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,"patient_stratified_shuffle(target_n=2,actual_n...",34,13,4,22,8


In [ ]:
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
IMAGENET_STD=torch.tensor([0.229,0.224,0.225]).view(3,1,1)

class LIDCDataset(Dataset):
    def __init__(self,df,training=False): self.df=df.reset_index(drop=True).copy(); self.training=training
    def __len__(self): return len(self.df)
    def _augment(self,x):
        # x: 1xHxW tensor in [0,1].
        if not self.training: return x
        if random.random()<0.5: x=torch.flip(x,dims=[2])
        if random.random()<0.5: x=torch.flip(x,dims=[1])
        angle=random.uniform(-15,15); max_shift=0.10*CONFIG['image_size']
        tx=random.uniform(-max_shift,max_shift); ty=random.uniform(-max_shift,max_shift)
        x=transforms.functional.affine(x,angle=angle,translate=[int(round(tx)),int(round(ty))],scale=1.0,shear=[0.0,0.0],fill=0.0)
        return x
    def __getitem__(self,i):
        r=self.df.iloc[i]; a=np.load(r.image_path).astype(np.float32,copy=False)
        x=torch.from_numpy(a).unsqueeze(0); x=self._augment(x)
        x=x.repeat(3,1,1); x=(x-IMAGENET_MEAN)/IMAGENET_STD
        item={'image':x,'label':torch.tensor(int(r.label),dtype=torch.long),'image_path':str(r.image_path),
              'patient_id':str(r.patient_id),'nodule_id':str(r.nodule_id)}
        # Future-compatible contour path; current preprocessing v3 does not create this field.
        if 'mask_path' in self.df.columns: item['mask_path']=str(r.mask_path) if pd.notna(r.mask_path) else ''
        return item

def make_loader(df,shuffle=False,batch_size=None,training=None):
    if training is None: training=bool(shuffle)
    return DataLoader(LIDCDataset(df,training=training),batch_size=batch_size or CONFIG['batch_size'],shuffle=shuffle,
                      num_workers=CONFIG['num_workers'],pin_memory=torch.cuda.is_available(),drop_last=False)


In [ ]:
class B3ResNet50EDL(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=resnet50(weights=ResNet50_Weights.DEFAULT)
        self.backbone.fc=nn.Linear(self.backbone.fc.in_features,CONFIG["num_classes"])
    def forward(self,x):
        return self.backbone(x)

def build_model():
    return B3ResNet50EDL().to(DEVICE)

In [ ]:

EPS=CONFIG["epsilon"]

def logits_to_dirichlet(logits,prior_C=None):
    C=CONFIG["prior_weight_C"] if prior_C is None else float(prior_C)
    if C <= 0:
        raise ValueError("prior_C must be > 0.")
    evidence=F.softplus(logits)
    alpha=evidence+C
    strength=alpha.sum(1,keepdim=True)
    probs=alpha/strength
    vacuity=(CONFIG["num_classes"]*C)/strength.squeeze(1)
    ent=-(probs*torch.log(probs.clamp_min(EPS))).sum(1)
    return {
        "evidence":evidence,"alpha":alpha,"strength":strength.squeeze(1),
        "probs":probs,"vacuity":vacuity,"aleatoric_entropy":ent,
        "normalized_aleatoric_entropy":ent/math.log(CONFIG["num_classes"]),
    }

def expected_dirichlet_ce(alpha,targets):
    y=F.one_hot(targets,num_classes=CONFIG["num_classes"]).float()
    S=alpha.sum(1,keepdim=True)
    return (y*(torch.digamma(S)-torch.digamma(alpha))).sum(1).mean()

def dirichlet_kl(alpha,beta):
    """KL[Dir(alpha) || Dir(beta)] for batched concentration vectors."""
    sa=alpha.sum(1,keepdim=True)
    sb=beta.sum(1,keepdim=True)
    logB_a=torch.lgamma(alpha).sum(1,keepdim=True)-torch.lgamma(sa)
    logB_b=torch.lgamma(beta).sum(1,keepdim=True)-torch.lgamma(sb)
    dig=((alpha-beta)*(torch.digamma(alpha)-torch.digamma(sa))).sum(1,keepdim=True)
    return (logB_b-logB_a+dig).mean()

def kl_anneal_factor(epoch):
    # B5: lambda_t = min(1, t/10), configurable and recorded.
    T=max(int(CONFIG["kl_anneal_epochs"]),1)
    return min(1.0,float(epoch)/float(T))

def vanilla_edl_loss(logits,targets,epoch,prior_C=None,lambda_kl=None,loss_mode="ece_plus_kl"):
    C=CONFIG["prior_weight_C"] if prior_C is None else float(prior_C)
    lkl=CONFIG["lambda_kl"] if lambda_kl is None else float(lambda_kl)
    d=logits_to_dirichlet(logits,C)
    alpha=d["alpha"]
    lece=expected_dirichlet_ce(alpha,targets)

    # C-consistent evidence-removal KL:
    # target-class concentration is reset to C, while non-target evidence is retained.
    y=F.one_hot(targets,num_classes=CONFIG["num_classes"]).float()
    alpha_tilde=y*C+(1-y)*alpha
    beta=torch.full_like(alpha_tilde,C)
    lkloss=dirichlet_kl(alpha_tilde,beta)

    anneal=kl_anneal_factor(epoch)
    effective_lambda=lkl*anneal
    total=lece if loss_mode=="ece_only" else lece+effective_lambda*lkloss
    return total,{
        "loss_total":float(total.detach()),
        "loss_ece":float(lece.detach()),
        "loss_kl":float(lkloss.detach()),
        "kl_anneal_factor":float(anneal),
        "effective_lambda_kl":float(effective_lambda),
    }


In [ ]:

# ---------------- Verification V1: analytic/unit checks ----------------
def _reference_dirichlet_kl(alpha,beta):
    qa=torch.distributions.Dirichlet(alpha)
    qb=torch.distributions.Dirichlet(beta)
    return torch.distributions.kl.kl_divergence(qa,qb).mean()

def run_b3_loss_unit_tests():
    # T2-style KL verification against torch.distributions.
    alpha=torch.tensor([[5.0,1.0,2.0],[0.5,3.0,0.7]],dtype=torch.float64)
    for C in [0.5,1.0,2.0]:
        beta=torch.full_like(alpha,C)
        ours=dirichlet_kl(alpha,beta)
        ref=_reference_dirichlet_kl(alpha,beta)
        if not torch.allclose(ours,ref,rtol=1e-9,atol=1e-9):
            raise AssertionError(f"Dirichlet KL mismatch for C={C}: ours={ours}, ref={ref}")

    # T3: with C != 1, removing target evidence must reset target alpha to C,
    # and the KL reference prior must also be Dir(C*1).
    targets=torch.tensor([0,1])
    for C in [0.5,2.0]:
        raw=torch.tensor([[1.2,-0.4,0.3],[-0.1,0.8,0.2]],dtype=torch.float64)
        d=logits_to_dirichlet(raw,C)
        y=F.one_hot(targets,num_classes=3).to(torch.float64)
        alpha_tilde=y*C+(1-y)*d["alpha"]
        assert torch.allclose(alpha_tilde[y.bool()],torch.full_like(alpha_tilde[y.bool()],C))

    # T5b-related guard for B3:
    # B3 is hard-label EDL (not the soft-label C1 case), but its C-consistent
    # KL must penalize unnecessary non-target evidence so strength cannot grow
    for C in [0.5, 1.0, 2.0]:
        y0 = torch.tensor([0])
        # Same target evidence, but much larger non-target evidence.
        a_low = torch.tensor([[5.0*C, C, C]], dtype=torch.float64)
        a_high = torch.tensor([[5.0*C, 20.0*C, 20.0*C]], dtype=torch.float64)
        yoh = F.one_hot(y0, num_classes=3).to(torch.float64)
        at_low = yoh*C + (1-yoh)*a_low
        at_high = yoh*C + (1-yoh)*a_high
        beta = torch.full_like(at_low, C)
        assert _reference_dirichlet_kl(at_high, beta) > _reference_dirichlet_kl(at_low, beta)

    # Annealing contract.
    assert kl_anneal_factor(1)==0.1
    assert kl_anneal_factor(5)==0.5
    assert kl_anneal_factor(10)==1.0
    assert kl_anneal_factor(50)==1.0
    print("B3 V1 loss tests: PASS")

run_b3_loss_unit_tests()


B3 V1 loss tests: PASS


In [ ]:
def brier(y,p):
    yy=np.eye(3)[np.asarray(y,dtype=int)]
    return float(np.mean(np.sum((np.asarray(p)-yy)**2,1)))

def nll(y,p):
    y=np.asarray(y,dtype=int); p=np.clip(np.asarray(p),1e-12,1)
    return float(-np.mean(np.log(p[np.arange(len(y)),y])))

def ece(y,p,bins=15):
    y=np.asarray(y); p=np.asarray(p)
    conf=p.max(1); pred=p.argmax(1); corr=pred==y
    edges=np.linspace(0,1,bins+1); out=0.0
    for i in range(bins):
        lo,hi=edges[i],edges[i+1]
        m=(conf>=lo)&(conf<=hi) if i==0 else (conf>lo)&(conf<=hi)
        if m.any(): out+=m.mean()*abs(corr[m].mean()-conf[m].mean())
    return float(out)

def safe_macro_auroc(y_true, probs):
    """T7-safe macro OvR AUROC shared by B1, B3, and C1."""
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probs, dtype=float)
    yb = label_binarize(y, classes=[0, 1, 2])

    aucs = []
    for k in range(3):
        # AUROC is undefined if this one-vs-rest target has only one value.
        if len(np.unique(yb[:, k])) < 2:
            continue
        aucs.append(float(roc_auc_score(yb[:, k], p[:, k])))

    return float(np.mean(aucs)) if aucs else np.nan

def metrics(y,p):
    pred=np.asarray(p).argmax(1)
    return {
        "accuracy":float(accuracy_score(y,pred)),
        "f1_macro":float(f1_score(y,pred,average="macro",zero_division=0)),
        "auroc_macro_ovr":safe_macro_auroc(y,p),
        "brier_score":brier(y,p),
        "ece_15":ece(y,p,15),
        "nll":nll(y,p),
    }

In [ ]:
@torch.no_grad()
def edl_predict(model,loader,prior_C=None):
    model.eval()
    out={k:[] for k in ["y_true","probs","vacuity","aleatoric_entropy","normalized_aleatoric_entropy","strength","evidence"]}
    paths=[]; patients=[]
    for b in loader:
        x=b["image"].to(DEVICE)
        d=logits_to_dirichlet(model(x),prior_C)
        out["y_true"].append(b["label"].numpy())
        for k in ["probs","vacuity","aleatoric_entropy","normalized_aleatoric_entropy","strength","evidence"]:
            out[k].append(d[k].cpu().numpy())
        paths.extend(b["image_path"]); patients.extend(b["patient_id"])
    for k in out: out[k]=np.concatenate(out[k])
    out["paths"]=paths; out["patients"]=patients
    return out

In [ ]:
def train_one_fold(fold,train_df,val_df,prior_C=None,lambda_kl=None,loss_mode="ece_plus_kl",run_tag="main"):
    seed_everything(CONFIG["seed"]+fold)
    model=build_model()
    opt=torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        betas=(CONFIG["adam_beta1"],CONFIG["adam_beta2"]),
        weight_decay=CONFIG["weight_decay"],
    )
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CONFIG["max_epochs"],eta_min=CONFIG["eta_min"])
    tr_loader=make_loader(train_df,True); va_loader=make_loader(val_df,False)

    C=CONFIG["prior_weight_C"] if prior_C is None else float(prior_C)
    lkl=CONFIG["lambda_kl"] if lambda_kl is None else float(lambda_kl)

    best=float("inf"); stale=0; hist=[]
    ckpt=CHECKPOINT_ROOT/f"fold_{fold}"/f"B3_{run_tag}_best.pt"

    for epoch in range(1,CONFIG["max_epochs"]+1):
        model.train(); totals={"loss_total":0.0,"loss_ece":0.0,"loss_kl":0.0}; seen=0; train_correct=0
        epoch_anneal=kl_anneal_factor(epoch)
        for b in tqdm(tr_loader,desc=f"B3 {run_tag} fold {fold} epoch {epoch}",leave=False):
            x=b["image"].to(DEVICE); y=b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits=model(x)
            loss,comp=vanilla_edl_loss(logits,y,epoch,C,lkl,loss_mode)
            loss.backward(); opt.step()
            bs=len(y); seen+=bs; train_correct += int((logits.argmax(1)==y).sum().item())
            for k in totals: totals[k]+=comp[k]*bs

        vr=edl_predict(model,va_loader,C)
        vm=metrics(vr["y_true"],vr["probs"])
        row={
            "fold":fold,"epoch":epoch,"prior_C":C,"lambda_kl":lkl,"loss_mode":loss_mode,
            "lr":opt.param_groups[0]["lr"],"kl_anneal_factor":epoch_anneal,"effective_lambda_kl":lkl*epoch_anneal,"train_accuracy":train_correct/max(seen,1),
            **{k:v/max(seen,1) for k,v in totals.items()},
            **{f"val_{k}":v for k,v in vm.items()}
        }
        hist.append(row)
        vb=vm["brier_score"]
        print(f"fold={fold} epoch={epoch} loss={row['loss_total']:.4f} val_Brier={vb:.4f} val_F1={vm['f1_macro']:.4f}")
        if vb<best-1e-8:
            best=vb; stale=0
            torch.save({
                "config":CONFIG,"fold":fold,"prior_C":C,"lambda_kl":lkl,
                "loss_mode":loss_mode,"epoch":epoch,"best_val_brier":best,
                "model_state_dict":model.state_dict()
            },ckpt)
        else:
            stale+=1
        sch.step()
        if stale>=CONFIG["early_stopping_patience"]:
            break

    torch.save({
        "config":CONFIG,"fold":fold,"prior_C":C,"lambda_kl":lkl,"loss_mode":loss_mode,
        "epoch":hist[-1]["epoch"] if hist else 0,"model_state_dict":model.state_dict()
    }, CHECKPOINT_ROOT/f"fold_{fold}"/f"B3_{run_tag}_final.pt")
    pd.DataFrame(hist).to_csv(LOG_ROOT/f"B3_{run_tag}_fold_{fold}_history.csv",index=False)
    state=torch.load(ckpt,map_location=DEVICE)
    model.load_state_dict(state["model_state_dict"])
    model.eval()
    return model,pd.DataFrame(hist),ckpt


In [ ]:


def rejection_curve(y, p, u):
    """Verifier-aligned full rejection curve using every retained count r=1,...,N."""
    y = np.asarray(y)
    p = np.asarray(p)
    u = np.asarray(u)

    n = len(y)
    order = np.argsort(u)  # low uncertainty retained first
    correct = (p.argmax(1) == y)[order].astype(np.float64)
    r = np.arange(1, n + 1)

    return pd.DataFrame({
        "coverage": r / n,
        "n_retained": r,
        "accuracy": np.cumsum(correct) / r
    })

def random_rejection(y, p, repeats=50, seed=42):
    """Random-rejection lower bound on the same full r=1,...,N coverage axis."""
    y = np.asarray(y)
    pred = np.asarray(p).argmax(1)
    n = len(y)
    r = np.arange(1, n + 1)
    rng = np.random.RandomState(seed)

    curves = []
    for _ in range(repeats):
        order = rng.permutation(n)
        correct = (pred[order] == y[order]).astype(np.float64)
        curves.append(np.cumsum(correct) / r)

    a = np.asarray(curves)
    return pd.DataFrame({
        "coverage": r / n,
        "n_retained": r,
        "accuracy": a.mean(0),
        "accuracy_sd": a.std(0)
    })

def curve_auc(df):
    s = df.sort_values("coverage")
    return float(np.trapezoid(
        s.accuracy.to_numpy(),
        s.coverage.to_numpy()
    ))

def efficiency(model,loader):
    params=sum(p.numel() for p in model.parameters())
    dummy=torch.randn(1,3,224,224,device=DEVICE)
    try: flops=float(profile(model,inputs=(dummy,),verbose=False)[0])
    except Exception: flops=np.nan
    one=DataLoader(loader.dataset,batch_size=1,shuffle=False,num_workers=0)
    times=[]
    for j,b in enumerate(one):
        if j>=min(30,len(loader.dataset)): break
        x=b["image"].to(DEVICE)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t=time.perf_counter()
        with torch.no_grad(): _=logits_to_dirichlet(model(x))
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append((time.perf_counter()-t)*1000)
    return {
        "total_parameters":int(params),
        "single_pass_flops":flops,
        "single_pass_latency_ms_per_image":float(np.mean(times)) if times else np.nan,
        "device":str(DEVICE),
    }


In [ ]:
# XAI implementation is defined here but intentionally executed only after all B3 training/evaluation cells.
class SaliencyEngine:
    def __init__(self,model):
        self.model=model; self.layer=model.backbone.layer4; self.a=None; self.g=None
        self.fh=self.layer.register_forward_hook(lambda m,i,o:setattr(self,'a',o))
        self.bh=self.layer.register_full_backward_hook(lambda m,gi,go:setattr(self,'g',go[0]))
    def remove(self): self.fh.remove(); self.bh.remove()
    def _heat(self,scalar):
        self.model.zero_grad(set_to_none=True); scalar.backward(); w=self.g.mean((2,3),keepdim=True)
        h=F.relu((w*self.a).sum(1,keepdim=True)); h=F.interpolate(h,size=(224,224),mode='bilinear',align_corners=False)[0,0].detach().cpu().numpy()
        h-=h.min(); h/=max(float(h.max()),CONFIG['epsilon']); return h
    def gradcam(self,x):
        logits=self.model(x); d=logits_to_dirichlet(logits); c=int(d['probs'].argmax(1).item()); return self._heat(d['probs'][:,c].sum()),c
    def esm_vacuity(self,x,C=None): return self._heat(logits_to_dirichlet(self.model(x),C)['vacuity'].sum())
    def esm_aleatoric(self,x,C=None): return self._heat(logits_to_dirichlet(self.model(x),C)['aleatoric_entropy'].sum())


In [ ]:
def _raw_patch_from_normalized_tensor(x):
    # Undo ImageNet normalization and return the original one-channel [0,1] CT patch.
    z=x.detach().cpu()[0]*IMAGENET_STD+IMAGENET_MEAN
    return z.mean(0).clamp(0,1).numpy()

def pointing_game(heat,mask):
    return int(bool(mask.reshape(-1)[int(np.argmax(heat))]))

def boundary_distance(heat,mask):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    boundary=np.logical_xor(mask.astype(bool),binary_erosion(mask.astype(bool)))
    if not boundary.any(): return np.nan
    dist=distance_transform_edt(~boundary); return float(dist[np.unravel_index(np.argmax(heat),heat.shape)])

def _xai_target_value(model,x,target_kind,C=None,target_class=None):
    """Scalar that a saliency method is intended to explain."""
    d=logits_to_dirichlet(model(x),C)
    if target_kind=='class_probability':
        if target_class is None:
            target_class=int(d['probs'].argmax(1).item())
        return float(d['probs'][0,int(target_class)].item())
    if target_kind=='vacuity':
        return float(d['vacuity'][0].item())
    if target_kind=='aleatoric_entropy':
        return float(d['normalized_aleatoric_entropy'][0].item())
    raise ValueError(f'Unknown XAI target_kind: {target_kind}')

def perturbation_auc(model,x,heat,mode='deletion',steps=20,C=None,
                     target_kind='class_probability',target_class=None):
    """Insertion/deletion AUC against the scalar actually explained by the map."""
    if mode not in {'insertion','deletion'}:
        raise ValueError('mode must be insertion or deletion')
    if target_kind=='class_probability' and target_class is None:
        with torch.no_grad():
            target_class=int(logits_to_dirichlet(model(x),C)['probs'].argmax(1).item())

    #order=np.argsort(np.asarray(heat).reshape(-1))[::-1]
    order = np.argsort(np.asarray(heat).reshape(-1))[::-1].copy()
    base=x.clone()
    blurred=F.avg_pool2d(x,11,stride=1,padding=5)
    vals=[]; fracs=np.linspace(0,1,steps+1)
    for frac in fracs:
        n=int(round(float(frac)*len(order)))
        idx=order[:n]
        cur=(base.clone() if mode=='deletion' else blurred.clone())
        flat=cur.view(1,3,-1)
        src=(blurred if mode=='deletion' else base).view(1,3,-1)
        flat[:,:,idx]=src[:,:,idx]
        with torch.no_grad():
            vals.append(_xai_target_value(model,cur,target_kind,C,target_class))
    return float(np.trapezoid(np.asarray(vals,dtype=float),fracs))

def run_xai_for_fold(model,test_df,fold,C=None,max_examples=None):
    ds=LIDCDataset(test_df,training=False); eng=SaliencyEngine(model); rows=[]
    n=min(len(ds),max_examples or CONFIG['xai_max_examples_per_fold'])
    contour_available='mask_path' in test_df.columns and test_df['mask_path'].fillna('').astype(str).map(lambda p: Path(p).is_file()).any()
    try:
        for i in tqdm(range(n),desc=f'XAI fold {fold}'):
            s=ds[i]; x=s['image'].unsqueeze(0).to(DEVICE); gc,pred=eng.gradcam(x); ev=eng.esm_vacuity(x,C); ea=eng.esm_aleatoric(x,C)
            stem=Path(s['image_path']).stem; gd=GRADCAM_ROOT/f'fold_{fold}'; ed=ESM_ROOT/f'fold_{fold}'; gd.mkdir(parents=True,exist_ok=True); ed.mkdir(parents=True,exist_ok=True)
            np.save(gd/f'{stem}_gradcam.npy',gc); np.save(ed/f'{stem}_esm_vacuity.npy',ev); np.save(ed/f'{stem}_esm_aleatoric.npy',ea)
            row={'fold':fold,'nodule_id':s['nodule_id'],'image_path':s['image_path'],'predicted_label':pred,
                 'gradcam_insertion_auc':perturbation_auc(model,x,gc,'insertion',CONFIG['xai_metric_steps'],C,'class_probability',pred),
                 'gradcam_deletion_auc':perturbation_auc(model,x,gc,'deletion',CONFIG['xai_metric_steps'],C,'class_probability',pred),
                 'esm_vacuity_insertion_auc':perturbation_auc(model,x,ev,'insertion',CONFIG['xai_metric_steps'],C,'vacuity'),
                 'esm_vacuity_deletion_auc':perturbation_auc(model,x,ev,'deletion',CONFIG['xai_metric_steps'],C,'vacuity'),
                 'esm_aleatoric_insertion_auc':perturbation_auc(model,x,ea,'insertion',CONFIG['xai_metric_steps'],C,'aleatoric_entropy'),
                 'esm_aleatoric_deletion_auc':perturbation_auc(model,x,ea,'deletion',CONFIG['xai_metric_steps'],C,'aleatoric_entropy')}
            mp=s.get('mask_path','')
            if mp and Path(mp).is_file():
                mask=np.load(mp).astype(bool); row.update({'gradcam_pointing_hit':pointing_game(gc,mask),'esm_pointing_hit':pointing_game(ev,mask),
                    'gradcam_boundary_distance_px':boundary_distance(gc,mask),'esm_boundary_distance_px':boundary_distance(ev,mask)})
            rows.append(row)
    finally: eng.remove()
    out=pd.DataFrame(rows); out.to_csv(XAI_METRICS/f'fold_{fold}_xai_metrics.csv',index=False)
    return out,contour_available


In [ ]:

def evaluate_fold(model,test_df,fold,C=None,tag='main'):
    loader=make_loader(test_df,False,training=False)
    r=edl_predict(model,loader,C)
    p=r['probs']; y=r['y_true']; pred=p.argmax(1)

    # Preserve V and u per sample
    meta_lookup=test_df.set_index('image_path',drop=False)
    V=np.asarray([float(meta_lookup.loc[path,'variance_normalized']) for path in r['paths']])
    nodule_ids=[str(meta_lookup.loc[path,'nodule_id']) for path in r['paths']]

    pred_df=pd.DataFrame({
        'fold':fold,'image_path':r['paths'],'nodule_id':nodule_ids,
        'patient_id':r['patients'],'true_label':y,'predicted_label':pred,
        'p_benign':p[:,0],'p_indeterminate':p[:,1],'p_malignant':p[:,2],
        'V':V,'u':r['vacuity'],'vacuity':r['vacuity'],
        'aleatoric_entropy':r['aleatoric_entropy'],
        'normalized_aleatoric_entropy':r['normalized_aleatoric_entropy'],
        'dirichlet_strength':r['strength']
    })
    pred_df.to_csv(PREDICTION_ROOT/f'B3_{tag}_fold_{fold}_predictions.csv',index=False)

    m=metrics(y,p)
    rd=rejection_curve(y,p,r['vacuity'])
    rr=random_rejection(y,p,CONFIG['random_rejection_repeats'],CONFIG['seed']+fold)
    rd.to_csv(UNCERTAINTY_METRICS/f'B3_{tag}_fold_{fold}_rejection_vacuity.csv',index=False)
    rr.to_csv(UNCERTAINTY_METRICS/f'B3_{tag}_fold_{fold}_rejection_random.csv',index=False)
    eff=efficiency(model,loader)
    row={
        'fold':fold,'prior_C':CONFIG['prior_weight_C'] if C is None else float(C),
        'lambda_kl':CONFIG['lambda_kl'],'kl_anneal_epochs':CONFIG['kl_anneal_epochs'],
        'fold_assignments_sha256':actual_fold_hash,
        'test_patches':len(test_df),'test_patients':test_df.patient_id.nunique(),**m,
        'rejection_auc_vacuity':curve_auc(rd),'rejection_auc_random':curve_auc(rr),**eff,
        'mean_vacuity':float(np.mean(r['vacuity'])),
        'mean_aleatoric_entropy':float(np.mean(r['aleatoric_entropy'])),
        'mean_dirichlet_strength':float(np.mean(r['strength']))
    }
    cm=confusion_matrix(y,pred,labels=[0,1,2])
    np.save(CLASSIFICATION_METRICS/f'B3_{tag}_fold_{fold}_confusion_matrix.npy',cm)
    with open(CLASSIFICATION_METRICS/f'B3_{tag}_fold_{fold}_classification_report.json','w') as f:
        json.dump(classification_report(
            y,pred,labels=[0,1,2],target_names=CONFIG['class_names'],
            output_dict=True,zero_division=0),f,indent=2)
    pd.DataFrame([{k:row[k] for k in ['fold','f1_macro','auroc_macro_ovr','accuracy']}]).to_csv(
        CLASSIFICATION_METRICS/f'B3_{tag}_fold_{fold}_metrics.csv',index=False)
    pd.DataFrame([{k:row[k] for k in ['fold','brier_score','ece_15','nll']}]).to_csv(
        CALIBRATION_METRICS/f'B3_{tag}_fold_{fold}_metrics.csv',index=False)
    pd.DataFrame([{k:row[k] for k in [
        'fold','mean_vacuity','mean_aleatoric_entropy','mean_dirichlet_strength',
        'rejection_auc_vacuity','rejection_auc_random']}]).to_csv(
        UNCERTAINTY_METRICS/f'B3_{tag}_fold_{fold}_summary.csv',index=False)
    pd.DataFrame([{'fold':fold,**eff}]).to_csv(
        EFFICIENCY_METRICS/f'B3_{tag}_fold_{fold}_efficiency.csv',index=False)
    return row


In [ ]:
RUN_ALL_FOLDS=True
all_rows=[]
if RUN_ALL_FOLDS:
    for fold in range(1,CONFIG['n_splits']+1):
        tr,va,te=fold_frames[fold]['train'],fold_frames[fold]['val'],fold_frames[fold]['test']
        model,_,_=train_one_fold(fold,tr,va,CONFIG['prior_weight_C'],CONFIG['lambda_kl'],'ece_plus_kl','main')
        all_rows.append(evaluate_fold(model,te,fold,CONFIG['prior_weight_C'],'main'))
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    fold_metrics=pd.DataFrame(all_rows); fold_metrics.to_csv(METRICS_ROOT/'B3_fold_metrics.csv',index=False); display(fold_metrics)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 87.1MB/s]


B3 main fold 1 epoch 1:   0%|          | 0/2 [00:00<?, ?it/s]

fold=1 epoch=1 loss=1.3298 val_Brier=0.6385 val_F1=0.4000


B3 main fold 1 epoch 2:   0%|          | 0/2 [00:00<?, ?it/s]

...

B3 main fold 5 epoch 35:   0%|          | 0/1 [00:00<?, ?it/s]

fold=5 epoch=35 loss=0.7341 val_Brier=0.4746 val_F1=0.4545


B3 main fold 5 epoch 36:   0%|          | 0/1 [00:00<?, ?it/s]

fold=5 epoch=36 loss=0.7338 val_Brier=0.4727 val_F1=0.4545


,fold,prior_C,lambda_kl,kl_anneal_epochs,fold_assignments_sha256,test_patches,test_patients,accuracy,f1_macro,auroc_macro_ovr,...,nll,rejection_auc_vacuity,rejection_auc_random,total_parameters,single_pass_flops,single_pass_latency_ms_per_image,device,mean_vacuity,mean_aleatoric_entropy,mean_dirichlet_strength
0,1,1.0,1.0,10,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,2,2,0.000000,0.000000,1.000000,...,1.098459,0.000000,0.000000,23514179,4.131701e+09,12.281337,cuda,0.629385,1.090841,4.766577
1,2,1.0,1.0,10,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,19,5,0.736842,0.282828,0.798677,...,0.943474,0.736343,0.690284,23514179,4.131701e+09,10.754461,cuda,0.615608,1.079079,4.875102
2,3,1.0,1.0,10,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,6,4,0.833333,0.454545,0.000000,...,1.024754,0.744444,0.706833,23514179,4.131701e+09,12.428694,cuda,0.611006,1.088500,4.910583
3,4,1.0,1.0,10,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,11,5,0.818182,0.300000,0.496296,...,1.016622,0.826728,0.742088,23514179,4.131701e+09,12.820676,cuda,0.614815,1.094064,4.880423
4,5,1.0,1.0,10,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,10,5,0.500000,0.222222,0.417738,...,1.052801,0.228135,0.432223,23514179,4.131701e+09,14.761967,cuda,0.564661,1.063826,5.319724


In [ ]:
if RUN_ALL_FOLDS and all_rows:
    df=pd.DataFrame(all_rows)
    summary=[]
    for c in df.columns:
        if c!="fold" and pd.api.types.is_numeric_dtype(df[c]):
            v=pd.to_numeric(df[c],errors="coerce")
            summary.append({
                "metric":c,
                "mean":float(v.mean()) if v.notna().any() else np.nan,
                "std":float(v.std(ddof=1)) if v.notna().sum()>1 else np.nan,
                "n_valid_folds":int(v.notna().sum()),
            })
    summary=pd.DataFrame(summary)
    summary.to_csv(METRICS_ROOT/"B3_5fold_mean_std.csv",index=False)
    display(summary)

,metric,mean,std,n_valid_folds
0,prior_C,1.000000e+00,0.000000,5
1,lambda_kl,1.000000e+00,0.000000,5
2,kl_anneal_epochs,1.000000e+01,0.000000,5
3,test_patches,9.600000e+00,6.348228,5
4,test_patients,4.200000e+00,1.303840,5
5,accuracy,5.776715e-01,0.349385,5
6,f1_macro,2.519192e-01,0.164825,5
7,auroc_macro_ovr,5.425423e-01,0.383060,5
8,brier_score,6.185752e-01,0.041696,5
9,ece_15,3.774285e-01,0.162456,5


In [ ]:

if RUN_ALL_FOLDS and all_rows:
    mean_f1=float(pd.DataFrame(all_rows)['f1_macro'].mean())
    is_pilot=CONFIG['pilot_max_patients'] is not None
    milestone={
        'configuration':'B3','milestone':'M2',
        'threshold_f1_macro':0.50,
        'observed_mean_f1_macro':mean_f1,
        'pilot':bool(is_pilot),
        'passed':None if is_pilot else bool(mean_f1>0.50),
        'note':(
            'Pilot metrics are diagnostic only and are not used for milestone pass/fail.'
            if is_pilot else
            'Full-run diagnostic milestone: B3 mean F1-Macro > 0.50.'
        )
    }
    with open(REPORT_ROOT/'B3_M2_milestone.json','w') as f:
        json.dump(milestone,f,indent=2)
    print(milestone)


{'configuration': 'B3', 'milestone': 'M2', 'threshold_f1_macro': 0.5, 'observed_mean_f1_macro': 0.25191919191919193, 'pilot': False, 'passed': False, 'note': 'Full-run diagnostic milestone: B3 mean F1-Macro > 0.50.'}


In [ ]:
RUN_PRIOR_C_ABLATION=True
RUN_LOSS_ABLATION=True

def run_prior_C_ablation():
    rows=[]
    fold=1
    tr,va,te=fold_frames[1]["train"],fold_frames[1]["val"],fold_frames[1]["test"]
    for C in CONFIG["ablation_prior_C_values"]:
        tag=f"ablation_C_{str(C).replace('.','p')}"
        model,_,_=train_one_fold(1,tr,va,C,CONFIG["lambda_kl"],"ece_plus_kl",tag)
        r=evaluate_fold(model,te,1,C,tag)
        r["ablation"]="prior_C"; r["value"]=C
        rows.append(r)
        del model; gc.collect()
    out=pd.DataFrame(rows)
    out.to_csv(ABLATION_ROOT/"B3_fold0_prior_C_ablation.csv",index=False)
    return out

def run_loss_ablation():
    rows=[]
    tr,va,te=fold_frames[1]["train"],fold_frames[1]["val"],fold_frames[1]["test"]
    for mode in CONFIG["ablation_loss_modes"]:
        tag=f"ablation_{mode}"
        model,_,_=train_one_fold(1,tr,va,CONFIG["prior_weight_C"],CONFIG["lambda_kl"],mode,tag)
        r=evaluate_fold(model,te,1,CONFIG["prior_weight_C"],tag)
        r["ablation"]="loss_component"; r["value"]=mode
        rows.append(r)
        del model; gc.collect()
    out=pd.DataFrame(rows)
    out.to_csv(ABLATION_ROOT/"B3_fold0_loss_isolation.csv",index=False)
    return out

if RUN_PRIOR_C_ABLATION:
    display(run_prior_C_ablation())
if RUN_LOSS_ABLATION:
    display(run_loss_ablation())


B3 ablation_C_0p5 fold 1 epoch 1:   0%|          | 0/2 [00:00<?, ?it/s]

fold=1 epoch=1 loss=1.4501 val_Brier=0.6280 val_F1=0.4000


B3 ablation_C_0p5 fold 1 epoch 2:   0%|          | 0/2 [00:00<?, ?it/s]

fold=1 epoch=2 loss=1.4364 val_Brier=0.6077 val_F1=0.4000


B3 ablation_C_0p5 fold 1 epoch 3:   0%|          | 0/2 [00:00<?, ?it/s]

...

B3 ablation_ece_plus_kl fold 1 epoch 49:   0%|          | 0/2 [00:00<?, ?it/s]

fold=1 epoch=49 loss=0.6820 val_Brier=0.4420 val_F1=0.4000


B3 ablation_ece_plus_kl fold 1 epoch 50:   0%|          | 0/2 [00:00<?, ?it/s]

fold=1 epoch=50 loss=0.6781 val_Brier=0.4443 val_F1=0.4000


,fold,prior_C,lambda_kl,kl_anneal_epochs,fold_assignments_sha256,test_patches,test_patients,accuracy,f1_macro,auroc_macro_ovr,...,rejection_auc_random,total_parameters,single_pass_flops,single_pass_latency_ms_per_image,device,mean_vacuity,mean_aleatoric_entropy,mean_dirichlet_strength,ablation,value
0,1,1.0,1.0,10,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,2,2,0.0,0.0,0.5,...,0.0,23514179,4.131701e+09,8.376211,cuda,0.549348,1.069331,5.466766,loss_component,ece_only
1,1,1.0,1.0,10,6579bbae4001911dbeb74213340e7846c3dcc2d03f915e...,2,2,0.0,0.0,1.0,...,0.0,23514179,4.131701e+09,7.440568,cuda,0.629385,1.090841,4.766577,loss_component,ece_plus_kl


In [ ]:
# XAI target ablation required by the plan: class score vs vacuity vs aleatoric entropy.
saliency_ablation=pd.DataFrame([
    {'target':'Grad-CAM','scalar':'predicted class logit','gradient':'dy_c/dA'},
    {'target':'ESM-vacuity','scalar':'u=K*C/S','gradient':'du/dA'},
    {'target':'ESM-aleatoric ablation','scalar':'H_al=-sum p log p','gradient':'dH_al/dA'},
])
saliency_ablation.to_csv(ABLATION_ROOT/'B3_saliency_target_ablation_manifest.csv',index=False)
display(saliency_ablation)


,target,scalar,gradient
0,Grad-CAM,predicted class logit,dy_c/dA
1,ESM-vacuity,u=K*C/S,du/dA
2,ESM-aleatoric ablation,H_al=-sum p log p,dH_al/dA


In [ ]:

def paired_wilcoxon_comparison(b3_csv,other_csv,metric,other_name="other_config"):
    """Descriptive fold-paired summary only.

    With only five paired folds, a two-sided exact Wilcoxon test cannot attain p<0.05.
    Final inferential comparisons should use pooled out-of-fold predictions with
    patient-clustered bootstrap confidence intervals (and DeLong for AUROC).
    """
    a=pd.read_csv(b3_csv)[["fold",metric]].rename(columns={metric:"B3"})
    b=pd.read_csv(other_csv)[["fold",metric]].rename(columns={metric:other_name})
    paired=a.merge(b,on="fold",how="inner").dropna()
    if len(paired)!=5:
        raise ValueError(f"Expected 5 paired folds; found {len(paired)}")
    diff=paired["B3"].values-paired[other_name].values
    return {
        "metric":metric,"n_pairs":5,
        "B3_mean":float(paired["B3"].mean()),
        f"{other_name}_mean":float(paired[other_name].mean()),
        "mean_difference_B3_minus_other":float(np.mean(diff)),
        "inferential_test":"NOT_PERFORMED",
        "reason":"Five-fold two-sided exact Wilcoxon cannot attain p<0.05; use patient-clustered OOF bootstrap."
    }


In [ ]:
manifest={
 'configuration':'B3','backbone':'ResNet50','uq_method':'Vanilla EDL','loss':'L_ECE + lambda_KL * L_KL',
 'prior_C':CONFIG['prior_weight_C'],'lambda_KL':CONFIG['lambda_kl'],'kl_anneal_epochs':CONFIG['kl_anneal_epochs'],'kl_target':'Dirichlet(C*1), C-consistent',
 'lambda_KL_note':'Numeric lambda_KL is not specified in the supplied plan/proposal; this notebook records the explicit run value (default 1.0).',
 'input':{'format':'.npy','shape':[224,224,1],'preprocessed_range':[0,1],'model_channels':3,'channel_policy':'replicate grayscale to RGB'},
 'preprocessing_rules':CONFIG['preprocessing_expected'],'augmentation':CONFIG['augmentation'],
 'cv':{'folds':5,'patient_level':True,'fold_assignments_sha256':actual_fold_hash,'validation_fraction_of_non_test_patients':0.10},
 'training':{'optimizer':'AdamW','lr':1e-4,'eta_min':1e-6,'weight_decay':0.01,'epochs':50,'patience':10,'early_stop_metric':'validation Brier'},
 'metrics':['F1-Macro','Macro AUROC OvR','Brier','ECE-15','NLL','vacuity rejection curve','latency','FLOPs'],
 'xai':['Grad-CAM','ESM-vacuity','ESM-aleatoric ablation','Insertion/Deletion AUC','Pointing Game when contour masks exist','Boundary distance when contour masks exist'],
 'xai_note':'Current preprocessing v3 does not persist patch-space contour masks, so contour-dependent XAI metrics must not be fabricated.',
 'seed':42,'pilot_max_patients':CONFIG['pilot_max_patients'],'device':str(DEVICE),'patients_used':int(metadata.patient_id.nunique()),'patches_used':int(len(metadata)),
 'scope':{'HAAL':False,'lambda_max':'not applicable','T_warmup':'not applicable','alignment_mask_used_in_B3_loss':False},
}
with open(REPORT_ROOT/'B3_experiment_manifest.json','w') as f: json.dump(manifest,f,indent=2)
print(json.dumps(manifest,indent=2))


{
  "configuration": "B3",
  "backbone": "ResNet50",
  "uq_method": "Vanilla EDL",
  "loss": "L_ECE + lambda_KL * L_KL",
  "prior_C": 1.0,
  "lambda_KL": 1.0,
  "kl_anneal_epochs": 10,
  "kl_target": "Dirichlet(C*1), C-consistent",
  "lambda_KL_note": "Numeric lambda_KL is not specified in the supplied plan/proposal; this notebook records the explicit run value (default 1.0).",
  "input": {
    "format": ".npy",
    "shape": [
      224,
      224,
      1
    ],
    "preprocessed_range": [
      0,
      1
    ],
    "model_channels": 3,
    "channel_policy": "replicate grayscale to RGB"
  },
  "preprocessing_rules": {
    "minimum_diameter_mm": 3.0,
    "minimum_radiologists": 3,
    "variance_normalizer": 4.0,
    "hu_low": -1000.0,
    "hu_high": 400.0,
    "patch_range": [
      0.0,
      1.0
    ]
  },
  "augmentation": {
    "horizontal_flip": true,
    "vertical_flip": true,
    "rotation_degrees": 15,
    "translation_fraction": 0.1,
    "elastic_deformation": false,
    "int

## Final stage — XAI
Contour-dependent metrics must use the **saved preprocessing consensus masks** generated
after physical 1-mm resampling / FOV cropping.

In [ ]:
RUN_XAI=True  # enable only after the five B3 checkpoints exist
if RUN_XAI:
    xai_all=[]; contour_status=[]
    for fold in range(1,CONFIG['n_splits']+1):
        ckpt=CHECKPOINT_ROOT/f'fold_{fold}'/'B3_main_best.pt'
        if not ckpt.exists(): raise FileNotFoundError(f'Missing trained checkpoint: {ckpt}')
        model=build_model(); state=torch.load(ckpt,map_location=DEVICE); model.load_state_dict(state['model_state_dict']); model.eval()
        out,has_contours=run_xai_for_fold(model,fold_frames[fold]['test'],fold,CONFIG['prior_weight_C'])
        xai_all.append(out); contour_status.append({'fold':fold,'patch_space_contours_available':bool(has_contours)})
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    xai_df=pd.concat(xai_all,ignore_index=True) if xai_all else pd.DataFrame()
    xai_df.to_csv(XAI_METRICS/'B3_xai_all_folds.csv',index=False)
    pd.DataFrame(contour_status).to_csv(XAI_METRICS/'contour_availability.csv',index=False)
    if not all(x['patch_space_contours_available'] for x in contour_status):
        print('NOTE: Pointing Game and boundary-distance require preprocessing to persist patch-space radiologist contour masks. Insertion/Deletion AUC and saved Grad-CAM/ESM maps are still valid.')
    display(xai_df.head())



XAI fold 1:   0%|          | 0/2 [00:00<?, ?it/s]

XAI fold 2:   0%|          | 0/19 [00:00<?, ?it/s]

XAI fold 3:   0%|          | 0/6 [00:00<?, ?it/s]

XAI fold 4:   0%|          | 0/11 [00:00<?, ?it/s]

XAI fold 5:   0%|          | 0/10 [00:00<?, ?it/s]

NOTE: Pointing Game and boundary-distance require preprocessing to persist patch-space radiologist contour masks. Insertion/Deletion AUC and saved Grad-CAM/ESM maps are still valid.


,fold,nodule_id,image_path,predicted_label,gradcam_insertion_auc,gradcam_deletion_auc,esm_vacuity_insertion_auc,esm_vacuity_deletion_auc,esm_aleatoric_insertion_auc,esm_aleatoric_deletion_auc
0,1,LIDC-IDRI-0004_S15_N000,/content/drive/MyDrive/Thesis_Work/LIDC_Thesis...,1,0.397320,0.426149,0.628536,0.600404,0.978007,0.993240
1,1,LIDC-IDRI-0015_S26_N000,/content/drive/MyDrive/Thesis_Work/LIDC_Thesis...,1,0.389534,0.366855,0.629568,0.623958,0.992994,0.991409
2,2,LIDC-IDRI-0001_S12_N000,/content/drive/MyDrive/Thesis_Work/LIDC_Thesis...,1,0.425332,0.414212,0.618978,0.614663,0.988840,0.986330
3,2,LIDC-IDRI-0003_S14_N001,/content/drive/MyDrive/Thesis_Work/LIDC_Thesis...,1,0.411872,0.407095,0.606115,0.620125,0.986286,0.987977
4,2,LIDC-IDRI-0003_S14_N002,/content/drive/MyDrive/Thesis_Work/LIDC_Thesis...,1,0.495069,0.432231,0.596849,0.592059,0.979565,0.958745
